# Netflix Content Analysis: EDA and Insights

This notebook performs the exploratory data analysis (EDA) for the Open IIT Data Analytics Hackathon. It uses the preprocessed data generated by `data_preprocessing.py` and the helper functions from the `analytics_code` directory to derive insights.

## 1. Setup and Data Loading

First, let's import the necessary libraries and load the processed data tables.

In [ ]:
import pandas as pd
import sys
from pathlib import Path

# Add analytics_code to path
sys.path.append(str(Path.cwd().parent))

from analytics_code.data_preprocessing import run_all as run_preprocessing
from analytics_code.visualization_functions import generate_visualization_portfolio
from analytics_code.statistical_analysis import chi_square_test, lag_correlation
from analytics_code.text_analysis import tfidf_top_terms, add_sentiment

PROCESSED_DIR = Path.cwd().parent / 'artifacts' / 'processed'

# Run preprocessing if data is not already processed
if not PROCESSED_DIR.exists():
    print('Running data preprocessing pipeline...')
    run_preprocessing()
    print('Preprocessing complete.')

# Load data
titles = pd.read_parquet(PROCESSED_DIR / 'titles.parquet')
genres = pd.read_parquet(PROCESSED_DIR / 'genres.parquet')
countries = pd.read_parquet(PROCESSED_DIR / 'countries.parquet')
people = pd.read_parquet(PROCESSED_DIR / 'people.parquet')

print('Data loaded successfully.')
titles.head()

## 2. Univariate Analysis

Let's explore some of the key features of the dataset.

In [ ]:
print('% of Movies vs TV Shows:')
print(titles['type'].value_counts(normalize=True) * 100)

print('\nTop 10 Genres:')
print(genres['genre'].value_counts().head(10))

print('\nTop 10 Countries:')
print(countries['country_std'].value_counts().head(10))

print('\nTop 10 Ratings:')
print(titles['rating'].value_counts().head(10))

## 3. Bivariate and Multivariate Analysis

Now, let's look at the relationships between different variables.

In [ ]:
# Genre vs Country
genre_country = pd.crosstab(genres['genre'], countries['country_std'])
print('Genre vs Country Crosstab (Top 5x5):')
print(genre_country.iloc[:5, :5])

## 4. Text Analysis

We can extract keywords and sentiment from the content descriptions.

In [ ]:
titles_with_sentiment = add_sentiment(titles)
merged_sentiment_genre = titles_with_sentiment.merge(genres, on='show_id')

print('Average sentiment by genre:')
print(merged_sentiment_genre.groupby('genre')['sentiment'].mean().sort_values(ascending=False).head(10))

print('\nTop keywords from descriptions (TF-IDF):')
top_terms = tfidf_top_terms(merged_sentiment_genre, group_col='genre', top_k=5)
print(top_terms.head(15))

## 5. Statistical Tests

In [ ]:
# Chi-square test: Is genre independent of country?
merged_genre_country = genres.merge(countries, on='show_id')
top_genres = genres['genre'].value_counts().head(5).index
top_countries = countries['country_std'].value_counts().head(5).index
filtered_df = merged_genre_country[
    merged_genre_country['genre'].isin(top_genres) & 
    merged_genre_country['country_std'].isin(top_countries)
]
chi2_result = chi_square_test(filtered_df, 'genre', 'country_std')
print(f"Chi-square test for genre and country independence: p-value = {chi2_result['p_value']:.3f}")
if chi2_result['p_value'] < 0.05:
    print('Result: We reject the null hypothesis; genre and country are likely dependent.')
else:
    print('Result: We fail to reject the null hypothesis; genre and country may be independent.')

# Correlation: imdb_rating vs netflix addition speed?
lag_corr_result = lag_correlation(titles)
if lag_corr_result:
    print(f"\nSpearman correlation between release year and addition lag: rho = {lag_corr_result['spearman_rho']:.3f}, p-value = {lag_corr_result['p_value']:.3f}")

## 6. Generate All Visualizations

Finally, let's run the script to generate the full portfolio of over 30 charts and save them to the `visualizations/` directory.

In [ ]:
print('Generating visualization portfolio...')
generate_visualization_portfolio()
print('All visualizations have been generated.')